# Score the WinGo 30S forecasts yourself

How often did the 1,000-feature model correctly forecast the next round? Run the two code cells in order to count hits for all four targets. This uses the saved historical forecasts and the Python standard library. In a local checkout it reads the bundled CSV. In Colab it downloads the same file from a pinned GitHub commit and verifies its checksum.

In [1]:
from pathlib import Path
from urllib.request import urlopen
import csv
import hashlib
import io

relative_path = Path("docs/evidence/vscode/thousand_test_predictions_5000.csv")
local_file = next((folder / relative_path for folder in (Path.cwd(), *Path.cwd().parents)
                   if (folder / relative_path).is_file()), None)
commit = "197c6a8538c008f11fd3efa99edde08ed334431f"
url = f"https://raw.githubusercontent.com/choprasaksham2007-a11y/wingo-30s-lab/{commit}/{relative_path.as_posix()}"
if local_file:
    data = local_file.read_bytes()
else:
    with urlopen(url, timeout=30) as response:
        data = response.read()
expected = "436ad6f1ee41ea333a7090e2e9d5fdd6356f218251b6a23d150fb244b6aa6b7b"
if hashlib.sha256(data).hexdigest() != expected:
    raise ValueError("The prediction file differs from the published snapshot.")
rounds = list(csv.DictReader(io.StringIO(data.decode("utf-8-sig"))))
print(f"{len(rounds):,} saved forecasts loaded.")

5,000 saved forecasts loaded.


## Count hits

The actual digit determines three more labels. Digits 5–9 are Big. Odd digits have Green as their primary colour. For exact colour the stored class numbers are Red (0), Green (1), Red + Violet (2), and Green + Violet (3). The prediction columns contain the model’s stored class choices.

In [2]:
# Change these bounds to inspect a different chronological slice, then rerun this cell.
start, stop = 0, len(rounds)
if not 0 <= start < stop <= len(rounds):
    raise ValueError("Choose a nonempty slice within the saved rounds.")
sample = rounds[start:stop]
hits = dict.fromkeys(["Exact digit", "Big/Small", "Primary colour", "Full colour set"], 0)
for row in sample:
    digit = int(row["actual_digit"])
    exact_colour = {0: 2, 5: 3}.get(digit, digit % 2)
    hits["Exact digit"] += int(row["digit_prediction"]) == digit
    hits["Big/Small"] += int(row["size_prediction"]) == int(digit >= 5)
    hits["Primary colour"] += int(row["primary_prediction"]) == digit % 2
    hits["Full colour set"] += int(row["colour_prediction"]) == exact_colour

print(f"{'Target':<18}{'Correct':>8}{'Accuracy':>11}")
for target, count in hits.items():
    print(f"{target:<18}{count:>8,}{count / len(sample):>10.2%}")

Target             Correct   Accuracy
Exact digit            438     8.76%
Big/Small            2,413    48.26%
Primary colour       2,476    49.52%
Full colour set      1,951    39.02%


## Interpret the result

Across all 5,000 rows the model got **2,413 Big/Small calls right**. The separate saved recent-100 rule got **2,584 right** on the same rounds. The explored historical data does not establish that the rule will keep its lead on new rounds.

Try `start, stop = 1000, 2000` to inspect the second block. The model reached **52.0%** in that block even though it finished at **48.26%** overall. Reporting just a favourable window changes the impression a reader gets.

[Baseline record](https://github.com/choprasaksham2007-a11y/wingo-30s-lab/blob/main/docs/evidence/vscode/thousand_verification.json) · [Methods and limits](https://github.com/choprasaksham2007-a11y/wingo-30s-lab/blob/main/docs/RESULTS_AND_LIMITATIONS.md)